In [ ]:
from pathlib import Path
import re

dataset_dir = Path("dataset")

folders = {
    "images": ".png",
    "depth": ".png",
    "info": ".txt",
    "masks": ".png",
    "pointcloud": ".ply",
    "labels": ".txt",
}
START_INDEX = 0

# Get indices from images folder
image_files = sorted((dataset_dir / "images").glob("img*.png"))

pattern = re.compile(r"img(\d+)")

indices = []
for f in image_files:
    m = pattern.search(f.stem)
    if m:
        indices.append(int(m.group(1)))

# Build mapping old_index -> new_index
mapping = {
    old_idx: new_idx
    for new_idx, old_idx in enumerate(sorted(indices), start=START_INDEX)
}

print(f"Shiffting {len(mapping)} files starting at {START_INDEX}")

# First pass: rename to temporary names to avoid collisions
for folder, ext in folders.items():
    folder_path = dataset_dir / folder

    for old_idx, new_idx in mapping.items():
        old_file = folder_path / f"img{old_idx:04d}{ext}"

        if old_file.exists():
            tmp_file = folder_path / f"tmp_{old_idx:04d}{ext}"
            old_file.rename(tmp_file)

# Second pass: rename temp -> final
for folder, ext in folders.items():
    folder_path = dataset_dir / folder

    for old_idx, new_idx in mapping.items():
        tmp_file = folder_path / f"tmp_{old_idx:04d}{ext}"

        if tmp_file.exists():
            new_file = folder_path / f"img{new_idx:04d}{ext}"
            tmp_file.rename(new_file)

print("Renumbering complete.")

Found 44 samples
Renumbering complete.


In [1]:
from pathlib import Path
import re
import shutil

src_dir = Path("dataset")
dst_dir = Path("dataset-466")

folders = {
    "images": ".png",
    "depth": ".png",
    "info": ".txt",
    "masks": ".png",
    "pointcloud": ".ply",
    "labels": ".txt",
}
START_INDEX = 466

# Create destination folders
for folder in folders:
    (dst_dir / folder).mkdir(parents=True, exist_ok=True)

# Get indices from images folder
image_files = sorted((src_dir / "images").glob("img*.png"))

pattern = re.compile(r"img(\d+)")

indices = []
for f in image_files:
    m = pattern.search(f.stem)
    if m:
        indices.append(int(m.group(1)))

# Build mapping old_index -> new_index
mapping = {
    old_idx: new_idx
    for new_idx, old_idx in enumerate(sorted(indices), start=START_INDEX)
}

print(f"Copying {len(mapping)} files to {dst_dir}")

# Copy + rename
for folder, ext in folders.items():
    for old_idx, new_idx in mapping.items():
        src_file = src_dir / folder / f"img{old_idx:04d}{ext}"
        dst_file = dst_dir / folder / f"img{new_idx:04d}{ext}"

        if src_file.exists():
            shutil.copy2(src_file, dst_file)

print("New dataset created safely.")

Copying 1101 files to dataset-466
New dataset created safely.


In [3]:
from pathlib import Path
import re

dataset_dir = Path("dataset-multiple-img-only-final/dataset")

folder = "images"
ext =".png"

START_INDEX = 1701

# Get indices from images folder
image_files = sorted((dataset_dir / folder).glob("img*.png"))

pattern = re.compile(r"img(\d+)")

indices = []
for f in image_files:
    m = pattern.search(f.stem)
    if m:
        indices.append(int(m.group(1)))

# Build mapping old_index -> new_index
mapping = {
    old_idx: new_idx
    for new_idx, old_idx in enumerate(sorted(indices), start=START_INDEX)
}

print(f"Shiffting {len(mapping)} files starting at {START_INDEX}")

# First pass: rename to temporary names to avoid collisions

# to temp
for old_idx in mapping:
    old_file = dataset_dir/ folder / f"img{old_idx:04d}{ext}"

    if old_file.exists():
        tmp_file = dataset_dir/ folder / f"tmp_{old_idx:04d}{ext}"
        old_file.rename(tmp_file)

# Second pass: rename temp -> final
for old_idx, new_idx in mapping.items():
    tmp_file = dataset_dir/ folder / f"tmp_{old_idx:04d}{ext}"

    if tmp_file.exists():
        new_file = dataset_dir/ folder / f"img{new_idx:04d}{ext}"
        tmp_file.rename(new_file)

print("Renumbering complete.")


Shiffting 110 files starting at 1701
Renumbering complete.


In [ ]:
import cv2
import re
from pathlib import Path

# Define paths (Adjust if your folders are named differently)
base_path = Path("dataset")
uncropped_dir = base_path / "images"
info_dir = base_path / "info"
output_dir = base_path / "fixed_cropped_images"  # New folder for safety
output_dir.mkdir(exist_ok=True)

def parse_crop_offsets(info_path):
    """Extracts top and left crop values from the info txt file."""
    text = info_path.read_text()
    # Looks for something like: left=250 right=520
    match = re.search(r"top=(\d+)\s+bottom=(\d+)\s+left=(\d+)\s+right=(\d+)", text)
    if match:
        return {
            "top": int(match.group(1)),
            "bottom": int(match.group(2)),
            "left": int(match.group(3)),
            "right": int(match.group(4))
        }
    return None

# Process all uncropped images
for img_path in uncropped_dir.glob("*.png"):
    img_name = img_path.stem
    info_path = info_dir / f"{img_name}.txt"
    
    if not info_path.exists():
        print(f"[SKIP] Missing info file for {img_name}")
        continue
        
    crop = parse_crop_offsets(info_path)
    if crop is None:
        print(f"[SKIP] Could not parse crop bounds for {img_name}")
        continue
        
    # Load original image
    img = cv2.imread(str(img_path))
    h, w = img.shape[:2]
    
    # Apply the exact same crop logic your main.py used
    top, bottom, left, right = crop["top"], crop["bottom"], crop["left"], crop["right"]
    
    # Determine slicing bounds
    y_end = h - bottom if bottom > 0 else h
    x_end = w - right if right > 0 else w
    
    cropped_img = img[top:y_end, left:x_end]
    
    # Save the new correctly cropped image
    cv2.imwrite(str(output_dir / f"{img_name}.png"), cropped_img)
    print(f"[FIXED] Cropped {img_name}.png to match mask dimensions.")